# Encoder Orbit 群结构诊断

这个 notebook 不训练 neural network，不保存 `outputs/` 或 `artifacts/`，也不把 `D(P_gE(x))` 当作群结构证据。它只看 encoder latent：`E(gx)`、`P_gE(x)`、`P_g^{-1}E(gx)`、token correspondence、orbit consistency，以及 train/test Procrustes channel action。

核心目标是比较 `EQ-VAE / SD-VAE / RAE-DINOv2 / RAE-MAE / RAE-SigLIP` 的 latent 在 `C4 + flip` 变换下是否有不同的变换机制。

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import torch

CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / "train_eqvae").exists() else CWD.parent
sys.path.insert(0, str(ROOT))

from baselines.dinov2_token_diagnostics import (
    C4_TRANSFORMS,
    diagnostic_figure,
    load_baseline_adapter,
    load_named_dataset,
    orbit_alignment_figure,
    orbit_consistency_figure,
    pick_dataset_images,
    run_baseline_diagnostics,
    run_train_test_procrustes,
    run_vit_position_embedding_study,
    run_mae_dinov2_mechanism_study,
    run_group_structure_study,
    run_geometry_response_atlas,
    token_correspondence_figure,
)

print(f"ROOT = {ROOT}")
print(f"CUDA = {torch.cuda.is_available()}, GPUs = {torch.cuda.device_count()}")

## 1. 数据、模型和变换

建议优先用 Caltech101、Imagenette 或自定义 `image_folder`。CIFAR10 太小，可以做 sanity check，但不适合肉眼判断 token correspondence。

In [ ]:
dataset_root = "/data/shared"
dataset_name = "caltech101"
dataset_split = "train"
dataset_path = ""     # dataset_name="image_folder" 时使用，例如 "/data/shared/imagenette2-320/train"
download_dataset = False

image_size = 256
count = 4
seed = 0
indices = None         # 例如 [0, 5, 9]；None 表示按 seed 随机

device_name = "cuda:0"
rae_repo_path = ROOT / "external/RAE"
rae_auto_clone = False
rae_auto_download = False
posterior = "mode"     # VAE 用 mode 更稳定；RAE 会忽略这个参数

model_keys = ("eqvae", "sdvae", "rae_dinov2", "rae_mae", "rae_siglip2")
transforms = ("rot90", "rot180", "rot270", "flip_h", "flip_v")
orbit_transforms = C4_TRANSFORMS
center = "sample"      # "sample" 去掉每张图的全局 token 均值；"none" 看完整 latent

active_model_key = "rae_dinov2"
active_transform = "rot90"
sample_index = 0

In [ ]:
dataset = load_named_dataset(
    dataset_name,
    root=dataset_root,
    split=dataset_split,
    download=download_dataset,
    dataset_path=dataset_path,
)
x, selected_indices = pick_dataset_images(dataset, count=count, seed=seed, indices=indices, image_size=image_size)
print({"selected_indices": selected_indices, "x_shape": tuple(x.shape), "model_keys": model_keys})

## 2. 跨模型 encoder-side 定量表

这里按模型顺序加载、计算、释放，避免一次把所有 RAE/VAE 都放进显存。`direct_error`、cosine 和 displacement 都只基于 encoder latent。

In [ ]:
metric_rows, procrustes_rows, orbit_rows = run_baseline_diagnostics(
    model_keys,
    x,
    device=device_name,
    rae_repo_path=rae_repo_path,
    rae_auto_clone=rae_auto_clone,
    rae_auto_download=rae_auto_download,
    posterior=posterior,
    transforms=transforms,
    orbit_transforms=orbit_transforms,
    center=center,
)

metrics_df = pd.DataFrame(metric_rows)
procrustes_df = pd.DataFrame(procrustes_rows)
orbit_df = pd.DataFrame(orbit_rows)

display(metrics_df)
display(orbit_df)
display(procrustes_df)

## 3. Train/Test Procrustes Diagnostic

这一步用 train split 拟合每个变换的正交 channel map `C_g`，再在 val/test split 上评估 `Err_P`、`Err_PC` 和 `Gain`。默认小规模用于交互；论文实验建议改成 train 512 / val 256 / test 256。

In [ ]:
procrustes_model_keys = model_keys
procrustes_train_count = 32
procrustes_val_count = 16
procrustes_test_count = 16
procrustes_batch_size = 8
procrustes_centers = ("none", "sample")

# 论文规模建议：train 512 / val 256 / test 256。
# procrustes_train_count = 512
# procrustes_val_count = 256
# procrustes_test_count = 256

save_train_test_result = False
save_train_test_json = None  # 例如 ROOT / "local_procrustes_result.json"；默认不保存

In [ ]:
tt_rows, law_rows, tt_split_indices = run_train_test_procrustes(
    procrustes_model_keys,
    dataset,
    device=device_name,
    rae_repo_path=rae_repo_path,
    rae_auto_clone=rae_auto_clone,
    rae_auto_download=rae_auto_download,
    posterior=posterior,
    transforms=transforms,
    centers=procrustes_centers,
    image_size=image_size,
    train_count=procrustes_train_count,
    val_count=procrustes_val_count,
    test_count=procrustes_test_count,
    seed=seed,
    batch_size=procrustes_batch_size,
    save=save_train_test_result,
    save_json_path=save_train_test_json,
)

tt_df = pd.DataFrame(tt_rows)
law_df = pd.DataFrame(law_rows)
print({split: len(indices) for split, indices in tt_split_indices.items()})
display(tt_df)
display(tt_df[tt_df["split"] == "test"])
display(law_df)

## 4. MAE vs DINOv2 Mechanism Study

这一步只比较 `rae_dinov2` 和 `rae_mae`，用来解释“MAE 的 channel Procrustes 更强但生成更差”这个现象。它会返回全维 Procrustes、functional group-law、PCA 子空间指标、PCA explained variance 和 distribution complexity。默认小规模用于交互；论文规模建议 train 1024 / test 512。

In [ ]:
mechanism_keys = ("rae_dinov2", "rae_mae")
mechanism_train_count = 128
mechanism_test_count = 64
mechanism_batch_size = 8
mechanism_centers = ("sample",)
mechanism_components = (3, 16, 64, 128, 256)
mechanism_transforms = ("rot90", "rot180", "rot270", "flip_h")

# 论文规模建议：train 1024 / test 512。
# mechanism_train_count = 1024
# mechanism_test_count = 512

save_mechanism_result = False
save_mechanism_json = None  # 例如 ROOT / "local_mae_dinov2_mechanism.json"；默认不保存

In [ ]:
mechanism = run_mae_dinov2_mechanism_study(
    dataset,
    keys=mechanism_keys,
    device=device_name,
    rae_repo_path=rae_repo_path,
    rae_auto_clone=rae_auto_clone,
    rae_auto_download=rae_auto_download,
    transforms=mechanism_transforms,
    centers=mechanism_centers,
    pca_component_counts=mechanism_components,
    image_size=image_size,
    train_count=mechanism_train_count,
    test_count=mechanism_test_count,
    seed=seed,
    batch_size=mechanism_batch_size,
    save=save_mechanism_result,
    save_json_path=save_mechanism_json,
)

full_mech_df = pd.DataFrame(mechanism["full_rows"])
functional_mech_df = pd.DataFrame(mechanism["functional_law_rows"])
pca_mech_df = pd.DataFrame(mechanism["pca_rows"])
pca_functional_df = pd.DataFrame(mechanism["pca_functional_rows"])
pca_curve_df = pd.DataFrame(mechanism["pca_curve_rows"])
distribution_df = pd.DataFrame(mechanism["distribution_rows"])

print({split: len(indices) for split, indices in mechanism["split_indices"].items()})
display(full_mech_df[full_mech_df["split"] == "test"])
display(functional_mech_df)
display(pca_mech_df[pca_mech_df["split"] == "test"])
display(pca_functional_df)
display(pca_curve_df)
display(distribution_df)

In [ ]:
# 可选：PCA 子空间下的 gain 曲线。
# 这个 cell 只画 notebook 内图像，不保存文件。
import matplotlib.pyplot as plt

plot_df = pca_mech_df[(pca_mech_df["split"] == "test") & (pca_mech_df["transform"] == "rot90")]
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
for model_name, group in plot_df.groupby("model"):
    group = group.sort_values("actual_k")
    ax.plot(group["actual_k"], group["gain"], marker="o", label=model_name)
ax.set_xscale("log")
ax.set_xlabel("PCA components")
ax.set_ylabel("test Procrustes gain")
ax.set_title("rot90 PCA-subspace gain")
ax.legend()
fig.tight_layout()

## 5. Position / Patch Embedding Diagnostic

这一步专门排查 ViT absolute positional embedding 和 patch projection 是否破坏直接空间等变。它只适用于 `rae_dinov2` 和 `rae_mae`，不会用于 EQ-VAE / SD-VAE。重点看：`patch_pre_pos` 是否已经很高、`patch_plus_pos` 是否进一步变高、以及去掉/旋转 position embedding 是否真的改善最终输出。


In [ ]:
posdiag_keys = ("rae_dinov2", "rae_mae")
posdiag_count = 4
posdiag_transforms = ("rot90", "rot180", "flip_h", "flip_v")
posdiag_hidden_indices = (0, 1, 3, 6, 12)


In [ ]:
posdiag = run_vit_position_embedding_study(
    dataset,
    keys=posdiag_keys,
    device=device_name,
    rae_repo_path=rae_repo_path,
    rae_auto_clone=rae_auto_clone,
    rae_auto_download=rae_auto_download,
    transforms=posdiag_transforms,
    image_size=image_size,
    count=posdiag_count,
    seed=seed,
    center=center,
    hidden_indices=posdiag_hidden_indices,
)

pos_stage_df = pd.DataFrame(posdiag["stage_rows"])
pos_intervention_df = pd.DataFrame(posdiag["intervention_rows"])
print("indices", posdiag["indices"])
stage_order = [
    "patch_pre_pos", "pos_only", "pos_symmetry_none", "patch_plus_pos",
    "hidden_0", "hidden_1", "hidden_3", "hidden_6", "hidden_12",
    "final_raw", "rae_normalized",
]
pos_stage_df["stage"] = pd.Categorical(pos_stage_df["stage"], categories=stage_order, ordered=True)
display(pos_stage_df.sort_values(["model", "transform", "stage"]))
display(pos_intervention_df.sort_values(["model", "transform", "mode"]))
display(pos_stage_df.pivot_table(index=["model", "stage"], columns="transform", values="error", observed=False))
display(pos_intervention_df.pivot_table(index=["model", "mode"], columns="transform", values="error"))


In [ ]:
# 可选：layerwise error 曲线。这个 cell 只画 notebook 内图像，不保存文件。
import matplotlib.pyplot as plt

plot_df = pos_stage_df[pos_stage_df["stage"].isin([
    "patch_pre_pos", "patch_plus_pos", "hidden_1", "hidden_3", "hidden_6", "hidden_12", "final_raw"
])].copy()
fig, axes = plt.subplots(1, len(posdiag_keys), figsize=(6 * len(posdiag_keys), 4), sharey=True)
if len(posdiag_keys) == 1:
    axes = [axes]
for ax, model_name in zip(axes, posdiag_keys):
    model_df = plot_df[plot_df["model"] == model_name]
    for transform, group in model_df.groupby("transform"):
        group = group.sort_values("stage")
        ax.plot(group["stage"].astype(str), group["error"], marker="o", label=transform)
    ax.set_title(model_name)
    ax.set_xlabel("stage")
    ax.tick_params(axis="x", rotation=45)
    ax.legend(fontsize=8)
axes[0].set_ylabel("relative equivariance error")
fig.tight_layout()


## 6. Layerwise Geometry Atlas

这一步把问题从 final latent 扩展到整条 ViT 表征链：patch projection、position 后、block 1/3/6/9/12、final raw token、RAE normalized latent。它同时报告 direct equivariance、per-transform linear alignability 和 generator power consistency。


In [ ]:
atlas_keys = ("rae_dinov2", "rae_mae")
atlas_train_count = 128
atlas_test_count = 64
atlas_batch_size = 8
atlas_centers = ("sample",)
atlas_mean_residual_centers = ("none",)
atlas_analysis_transforms = ("rot90", "flip_h")
atlas_hidden_indices = (0, 1, 3, 6, 9, 12)
atlas_stage_names = (
    "patch_pre_pos", "patch_plus_pos", "hidden_1", "hidden_3", "hidden_6",
    "hidden_9", "hidden_12", "final_raw", "rae_normalized",
)
atlas_pca_components = (8, 16, 32, 64, 128, 256)

# 论文规模建议：train 1024 / test 512。
# atlas_train_count = 1024
# atlas_test_count = 512

save_atlas_result = False
save_atlas_json = None  # 例如 ROOT / "local_geometry_response_atlas.json"；默认不保存


In [ ]:
atlas = run_geometry_response_atlas(
    dataset,
    keys=atlas_keys,
    device=device_name,
    rae_repo_path=rae_repo_path,
    rae_auto_clone=rae_auto_clone,
    rae_auto_download=rae_auto_download,
    stage_names=atlas_stage_names,
    hidden_indices=atlas_hidden_indices,
    analysis_transforms=atlas_analysis_transforms,
    centers=atlas_centers,
    mean_residual_centers=atlas_mean_residual_centers,
    pca_component_counts=atlas_pca_components,
    image_size=image_size,
    train_count=atlas_train_count,
    test_count=atlas_test_count,
    position_count=posdiag_count,
    seed=seed,
    batch_size=atlas_batch_size,
    save=save_atlas_result,
    save_json_path=save_atlas_json,
)

layer_direct_df = pd.DataFrame(atlas["layer_direct_rows"])
layer_procrustes_df = pd.DataFrame(atlas["layer_procrustes_rows"])
layer_power_df = pd.DataFrame(atlas["layer_power_rows"])
mean_residual_df = pd.DataFrame(atlas["mean_residual_rows"])
pca_subspace_df = pd.DataFrame(atlas["pca_subspace_rows"])
atlas_pca_curve_df = pd.DataFrame(atlas["pca_curve_rows"])
atlas_position_df = pd.DataFrame(atlas["position_rows"])
atlas_position_intervention_df = pd.DataFrame(atlas["position_intervention_rows"])

print({split: len(indices) for split, indices in atlas["split_indices"].items()})
print("position_indices", atlas["position_indices"])
display(layer_direct_df[layer_direct_df["split"] == "test"])
display(layer_procrustes_df[layer_procrustes_df["split"] == "test"])
display(layer_power_df[(layer_power_df["split"] == "test") & (layer_power_df["transform"].isin(["rot180", "rot270"]))])
display(pca_subspace_df[(pca_subspace_df["split"] == "test") & (pca_subspace_df["transform"].isin(["rot180", "rot270"]))])


In [ ]:
# 可选：layerwise direct / Procrustes / power 曲线。这个 cell 只画 notebook 内图像，不保存文件。
import matplotlib.pyplot as plt

stage_order = list(atlas_stage_names)
plot_direct = layer_direct_df[layer_direct_df["split"] == "test"].copy()
plot_proc = layer_procrustes_df[layer_procrustes_df["split"] == "test"].copy()
plot_power = layer_power_df[(layer_power_df["split"] == "test") & (layer_power_df["transform"].isin(["rot180", "rot270"]))].copy()
for df in (plot_direct, plot_proc, plot_power):
    if "stage" in df:
        df["stage"] = pd.Categorical(df["stage"], categories=stage_order, ordered=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharex=True)
for (model_name, transform), group in plot_direct.groupby(["model", "transform"]):
    group = group.sort_values("stage")
    axes[0].plot(group["stage"].astype(str), group["err_p"], marker="o", label=f"{model_name} {transform}")
axes[0].set_title("direct equivariance")
axes[0].set_ylabel("Err_P")

for (model_name, transform), group in plot_proc.groupby(["model", "transform"]):
    group = group.sort_values("stage")
    axes[1].plot(group["stage"].astype(str), group["gain"], marker="o", label=f"{model_name} {transform}")
axes[1].set_title("linear alignability")
axes[1].set_ylabel("Procrustes gain")

for (model_name, transform), group in plot_power.groupby(["model", "transform"]):
    group = group.sort_values("stage")
    axes[2].plot(group["stage"].astype(str), group["power_over_ind"], marker="o", label=f"{model_name} {transform}")
axes[2].axhline(1.25, color="black", linestyle=":", linewidth=1)
axes[2].set_title("group consistency")
axes[2].set_ylabel("Err_power / Err_ind")

for ax in axes:
    ax.tick_params(axis="x", rotation=45)
    ax.legend(fontsize=7)
fig.tight_layout()


## 7. Mean vs Residual

`token_mean` 近似图像级语义；`spatial_residual` 是去掉 token mean 后的空间结构。这里不要把二者混成一个 latent 指标。


In [ ]:
mean_linear_df = mean_residual_df[mean_residual_df["diagnostic"] == "linear_alignability"]
mean_power_df = mean_residual_df[mean_residual_df["diagnostic"] == "group_consistency"]
display(mean_linear_df[mean_linear_df["split"] == "test"])
display(mean_power_df[(mean_power_df["split"] == "test") & (mean_power_df["transform"].isin(["rot180", "rot270"]))])

summary_rows = []
direct_summary = layer_direct_df[layer_direct_df["split"] == "test"].copy()
for (model_name, stage), group in direct_summary.groupby(["model", "stage"]):
    summary_rows.append({
        "model": model_name,
        "stage": stage,
        "direct_mean": group["err_p"].mean(),
    })
summary_df = pd.DataFrame(summary_rows)
display(summary_df.pivot(index="stage", columns="model", values="direct_mean"))


## 8. True Group Structure Test

这一步只拟合生成元 `C_90` 和 `C_flip_h`，再用 `C_90^k` 与 `srs` 去预测目标变换。它区分三件事：直接空间等变 `err_p`、独立拟合的单变换线性对齐 `err_ind`、由生成元组合得到的群表示预测 `err_power/err_relation`。


In [ ]:
group_keys = ("rae_dinov2", "rae_mae")
# 可选对照：group_keys = ("eqvae", "rae_dinov2", "rae_mae", "rae_siglip2")
group_train_count = 128
group_test_count = 64
group_batch_size = 8
group_centers = ("sample",)
group_components = (3, 16, 64, 128, 256)
group_generator_transforms = ("rot90", "flip_h")
group_independent_transforms = ("identity", "rot90", "rot180", "rot270", "flip_h")
group_power_transforms = C4_TRANSFORMS

# 论文规模建议：train 1024 / test 512。
# group_train_count = 1024
# group_test_count = 512

save_group_result = False
save_group_json = None  # 例如 ROOT / "local_true_group_structure.json"；默认不保存


In [ ]:
group_structure = run_group_structure_study(
    dataset,
    keys=group_keys,
    device=device_name,
    rae_repo_path=rae_repo_path,
    rae_auto_clone=rae_auto_clone,
    rae_auto_download=rae_auto_download,
    generator_transforms=group_generator_transforms,
    independent_transforms=group_independent_transforms,
    power_transforms=group_power_transforms,
    orbit_transforms=C4_TRANSFORMS,
    centers=group_centers,
    pca_component_counts=group_components,
    image_size=image_size,
    train_count=group_train_count,
    test_count=group_test_count,
    seed=seed,
    batch_size=group_batch_size,
    save=save_group_result,
    save_json_path=save_group_json,
)

power_df = pd.DataFrame(group_structure["power_rows"])
d4_df = pd.DataFrame(group_structure["d4_relation_rows"])
closure_df = pd.DataFrame(group_structure["orbit_closure_rows"])
pca_power_df = pd.DataFrame(group_structure["pca_power_rows"])
pca_d4_df = pd.DataFrame(group_structure["pca_d4_relation_rows"])
pca_closure_df = pd.DataFrame(group_structure["pca_orbit_closure_rows"])
group_pca_curve_df = pd.DataFrame(group_structure["pca_curve_rows"])

print({split: len(indices) for split, indices in group_structure["split_indices"].items()})
test_power = power_df[(power_df["split"] == "test") & (power_df["transform"] != "identity")]
test_d4 = d4_df[d4_df["split"] == "test"]
test_pca_power = pca_power_df[(pca_power_df["split"] == "test") & (pca_power_df["transform"].isin(["rot180", "rot270"]))]
test_pca_d4 = pca_d4_df[pca_d4_df["split"] == "test"]

display(test_power[["model", "center", "transform", "err_p", "err_ind", "err_power", "power_over_ind", "power_gain"]])
display(test_d4[["model", "center", "relation", "target_transform", "err_p", "err_ind", "err_relation", "relation_over_ind"]])
display(test_pca_power[["model", "center", "requested_k", "actual_k", "transform", "err_p", "err_ind", "err_power", "power_over_ind"]])
display(test_pca_d4[["model", "center", "requested_k", "actual_k", "relation", "err_relation", "relation_over_ind"]])
display(group_pca_curve_df)


In [ ]:
# 可选：真正群结构的两类图。这个 cell 只画 notebook 内图像，不保存文件。
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_df = test_power[test_power["transform"].isin(["rot90", "rot180", "rot270"])]
for model_name, group in plot_df.groupby("model"):
    group = group.set_index("transform").reindex(["rot90", "rot180", "rot270"])
    axes[0].plot(group.index, group["err_ind"], marker="o", label=f"{model_name} independent")
    axes[0].plot(group.index, group["err_power"], marker="s", linestyle="--", label=f"{model_name} power")
axes[0].set_ylabel("relative error")
axes[0].set_title("independent C_g vs generator power")
axes[0].legend(fontsize=8)

ratio_df = test_pca_power.copy()
for (model_name, transform), group in ratio_df.groupby(["model", "transform"]):
    group = group.sort_values("actual_k")
    axes[1].plot(group["actual_k"], group["power_over_ind"], marker="o", label=f"{model_name} {transform}")
axes[1].axhline(1.25, color="black", linestyle=":", linewidth=1)
axes[1].set_xscale("log")
axes[1].set_xlabel("PCA components")
axes[1].set_ylabel("Err_power / Err_ind")
axes[1].set_title("PCA-subspace power consistency")
axes[1].legend(fontsize=8)
fig.tight_layout()

heatmap_model = group_keys[0]
heatmap_df = closure_df[(closure_df["split"] == "test") & (closure_df["space"] == "full") & (closure_df["model"] == heatmap_model)]
matrix = heatmap_df.pivot(index="source_transform", columns="target_transform", values="closure_error")
matrix = matrix.reindex(index=list(C4_TRANSFORMS), columns=list(C4_TRANSFORMS))
fig, ax = plt.subplots(1, 1, figsize=(4.6, 4))
im = ax.imshow(matrix.values, cmap="magma")
ax.set_xticks(range(len(C4_TRANSFORMS)), C4_TRANSFORMS, rotation=45, ha="right")
ax.set_yticks(range(len(C4_TRANSFORMS)), C4_TRANSFORMS)
ax.set_xlabel("target")
ax.set_ylabel("source")
ax.set_title(f"{heatmap_model} C4 orbit closure")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()


## 9. 单模型四类可视化

下面只加载 `active_model_key`。这四类图分别看：单变换 alignment、完整 C4 orbit alignment、token correspondence、orbit consistency matrix。

In [ ]:
adapter = load_baseline_adapter(
    active_model_key,
    device=device_name,
    rae_repo_path=rae_repo_path,
    rae_auto_clone=rae_auto_clone,
    rae_auto_download=rae_auto_download,
    posterior=posterior,
)
print({"active_model_key": active_model_key, "device": str(adapter.device), "transform": active_transform})

In [ ]:
diagnostic_figure(adapter, x, transform=active_transform, sample_index=sample_index, center=center)

In [ ]:
orbit_alignment_figure(adapter, x, orbit_transforms=orbit_transforms, sample_index=sample_index)

In [ ]:
token_correspondence_figure(adapter, x, transform=active_transform, sample_index=sample_index, center=center)

In [ ]:
orbit_consistency_figure(adapter, x, orbit_transforms=orbit_transforms, center=center, sample_index=sample_index)

## 10. 读图准则

- `flip_h/flip_v` 强、`rot90` 弱：说明有部分空间群响应，但不是完整 C4 等变。
- encoder 指标好但 decoder 图差：说明 latent 可能有结构，decoder 对 transformed/OOD latent 不鲁棒。
- Train/test Procrustes gain 高：说明 channel action 可能是真实结构，而不是同批样本过拟合。
- Procrustes gain 高但 group-law error 高：说明可以线性拟合，但还不是稳定群表示。
- orbit consistency off-diagonal 高：说明同一图像的旋转轨道无法被固定空间作用稳定对齐。
- `err_ind` 低但 `err_power` 高：说明只是单变换线性可对齐，不是稳定群表示。
- `power_over_ind <= 1.25` 且 D4 relation error 小：才可以谨慎称为近似离散群结构。
- `C_90^4` 或 `srs=r^-1` functional error 高：说明生成元组合不能闭合，后续应转向 adapter/decomposition，而不是继续夸大 Procrustes gain。

- `patch_pre_pos` 已高：patch projection 本身就不满足旋转/翻转等变。
- `patch_plus_pos` 明显高于 `patch_pre_pos`：absolute positional embedding 是额外破坏源。
- `zero_pos_both` 或 `rotated_pos_for_gx` 只用于诊断；若不改善，说明 pretrained transformer 已强依赖固定坐标系，不能直接靠改 pos 得到群表示。

- Layerwise direct error 下降：说明 transformer block 正在把局部几何差异吸收到语义/上下文结构里。
- Layerwise Procrustes gain 高但 `power_over_ind` 高：说明该层有单变换线性可对齐，但还不是稳定群表示。
- `token_mean` 更稳定而 `spatial_residual` 更等变：说明全局语义和空间几何应分开建模。
